# CineEmbed — 03 Train Contrastive (Phase 1 sweep)

Self-supervised pretext stage that pulls together two modality-dropout views of the same film and pushes apart different films. After pretext the **backbone** is saved (projection head discarded per Chen et al. 2020) and is meant to be loaded by `02_train_ae.ipynb` / `04_train_dec.ipynb` for fine-tuning.

Spec: `docs/superpowers/specs/2026-05-06-clustering-improvement-techniques.md` §2.1.

**Sweep grid (3 configs, ~30 min each on T4):**

| run_name | tau | drop_prob | proj_dim |
|---|---|---|---|
| `contrastive_tau0p1_drop0p3` | 0.1 | 0.3 | 128 |
| `contrastive_tau0p5_drop0p3` | 0.5 | 0.3 | 128 |
| `contrastive_tau0p1_drop0p4` | 0.1 | 0.4 | 128 |

**Outputs (per run, under `<artifacts>/models/<run_name>/`):**
- `pretext_backbone.pt` — load this into `MultiModalBackbone` for fine-tune
- `pretext_full.pt` — backbone + projection, for resume
- `history.json`, `eval.json`

Run `00_colab_setup.ipynb` once per session first.

In [ ]:
import os, sys, json
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    try:
        from google.colab import userdata
        _token = userdata.get('GITHUB_TOKEN')
        REPO_URL = f"https://{_token}@github.com/barandincoguz/CineEmbed-.git"
    except Exception:
        REPO_URL = "https://github.com/barandincoguz/CineEmbed-.git"
    REPO_ROOT = Path('/content/cineembed-repo')
    ARTIFACTS = Path('/content/drive/MyDrive/CineEmbed/artifacts')
    if not REPO_ROOT.exists():
        get_ipython().system(f'git clone {REPO_URL} {REPO_ROOT}')
    else:
        get_ipython().system(f'cd {REPO_ROOT} && git fetch -q && git checkout feature/wandb-integration -q && git pull -q')
    get_ipython().system(f'pip install -e {REPO_ROOT} -q')
    get_ipython().system('pip install wandb -q')
else:
    REPO_ROOT = Path('..').resolve()
    ARTIFACTS = REPO_ROOT / 'artifacts'

sys.path.insert(0, str(REPO_ROOT / 'src'))

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Repo:      {REPO_ROOT}")
print(f"Artifacts: {ARTIFACTS}")
print(f"Device:    {DEVICE}")

# Sanity-check artifacts before launching expensive runs
for p in [ARTIFACTS / 'feature_matrix.npz', ARTIFACTS / 'movies_eda_final.csv']:
    assert p.exists(), f"Missing artifact: {p}"

## Optional: W&B login
Skip the next cell (or set `WANDB_PROJECT = None`) to train without W&B logging.

In [ ]:
WANDB_PROJECT = 'cineembed'  # set to None to disable

if WANDB_PROJECT and IN_COLAB:
    try:
        from google.colab import userdata
        os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    except Exception:
        print('No WANDB_API_KEY in Colab Secrets — will fall back to interactive login.')
    get_ipython().system('wandb login --relogin $WANDB_API_KEY 2>&1 | tail -3')

## Sweep — 3 configs
Each config invokes `scripts/train_contrastive.py` in a subprocess for clean isolation and per-run wandb context. With early-stop patience 8 and batch 1024 each run should converge in ~25-40 min on T4.

In [ ]:
SWEEP = [
    {'run_name': 'contrastive_tau0p1_drop0p3', 'tau': 0.1, 'drop_prob': 0.3, 'proj_dim': 128},
    {'run_name': 'contrastive_tau0p5_drop0p3', 'tau': 0.5, 'drop_prob': 0.3, 'proj_dim': 128},
    {'run_name': 'contrastive_tau0p1_drop0p4', 'tau': 0.1, 'drop_prob': 0.4, 'proj_dim': 128},
]

EPOCHS = 60
BATCH_SIZE = 1024
PATIENCE = 8

SCRIPT = REPO_ROOT / 'scripts' / 'train_contrastive.py'
assert SCRIPT.exists(), f"Missing {SCRIPT}"

for cfg in SWEEP:
    print('\n' + '#' * 78)
    print(f"# {cfg['run_name']}")
    print('#' * 78)
    cmd = (
        f"python {SCRIPT}"
        f" --artifacts {ARTIFACTS}"
        f" --run-name {cfg['run_name']}"
        f" --tau {cfg['tau']} --drop-prob {cfg['drop_prob']} --proj-dim {cfg['proj_dim']}"
        f" --batch-size {BATCH_SIZE} --epochs {EPOCHS} --patience {PATIENCE}"
        f" --device {DEVICE}"
    )
    if WANDB_PROJECT:
        cmd += f" --wandb-project {WANDB_PROJECT}"
    print('>', cmd)
    rc = get_ipython().system(cmd)
    if rc:
        print(f"!! {cfg['run_name']} returned non-zero: {rc}")

## Summary — collect eval.json from each run

In [ ]:
rows = []
for cfg in SWEEP:
    p = ARTIFACTS / 'models' / cfg['run_name'] / 'eval.json'
    if not p.exists():
        print(f"  (missing eval.json for {cfg['run_name']})")
        continue
    e = json.loads(p.read_text())
    km = e.get('kmeans_k21', {})
    gm = e.get('gmm_k21', {})
    pa = e.get('per_axis_k_kmeans', {})
    rows.append({
        'run':            cfg['run_name'],
        'tau':            cfg['tau'],
        'drop':           cfg['drop_prob'],
        'km_gNMI':        km.get('genre_nmi'),
        'km_dNMI':        km.get('decade_nmi'),
        'km_lNMI':        km.get('lang_nmi'),
        'gmm_gNMI':       gm.get('genre_nmi'),
        'per_axis_gNMI':  pa.get('genre_nmi_k21'),
        'per_axis_dNMI':  pa.get('decade_nmi_k12'),
        'per_axis_lNMI':  pa.get('lang_nmi_k11'),
    })

try:
    import pandas as pd
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
except ImportError:
    for r in rows:
        print(r)

print('\nMVP baseline (for comparison): dec_z64_k21 genre_NMI = 0.332')

## Next step — fine-tune AE/DEC on top of best pretext backbone

Pick the row with the highest `km_gNMI` (or `gmm_gNMI`) and pass its `pretext_backbone.pt` into `02_train_ae.ipynb` by loading the state dict before training:

```python
bb = backbone.MultiModalBackbone(BLOCK_DIMS, PROJ_DIMS, hidden_dim=128, latent_dim=64)
bb.load_state_dict(torch.load(ARTIFACTS / 'models' / '<best-run>' / 'pretext_backbone.pt'))
head = heads.AEHead(bb, BLOCK_DIMS, PROJ_DIMS, hidden_dim=128)
# … then continue with the existing AE training loop
```

Expected payoff per spec §2.1: +5–12% NMI vs cold-start (MVP baseline 0.332).